## Regularize experiment results summary
This notebook compares results from all regularize experiments

## Setup
Set up file system for the datset(using Google Drive), dagshub and MLflow

In [ ]:
# Install dependencies
%%capture
%pip install -q dagshub[jupyter]
!pip install mlflow

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

# Set up dagshub
!dagshub login
import dagshub
TOKEN = dagshub.auth.get_token()

## Compare Experiment Results

In [ ]:
import os
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, roc_auc_score

# Define base directory
base_dir = "/content/drive/MyDrive/regularize-experiments"

# Define experiments to analyze
experiment_dirs = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])

# Dictionary to store results
experiment_results = []

# Reload test set (ensuring consistency across evaluations)
data_dir = "/content/drive/MyDrive/Omdena/urban-green-frankfurt/MULC"
image_dir = "VBWVA_8R"
X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(*read_file(data_dir, image_dir, multiclass=False))
del X_train, X_val, y_train, y_val  # Free up memory

# Loop through each experiment
for experiment in experiment_dirs:
    exp_dir = os.path.join(base_dir, experiment)
    model_path = os.path.join(exp_dir, "unet.keras")
    history_path = os.path.join(exp_dir, "unet_training_history.pkl")

    # Load training history
    with open(history_path, "rb") as f:
        history = pickle.load(f)

    # Load model
    model = load_model(model_path, compile=False)  # Load without compiling

    # Determine optimizer type
    if experiment == "experiment07_add_weigth_decay":
        optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-4)
    elif experiment == "experiment08_decrease_weigth_decay":
        optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=5e-5)
    elif experiment == "experiment09_increase_weigth_decay":
        optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=5e-4)
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

    # Recompile with the correct metrics
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.BinaryFocalCrossentropy(from_logits=True),
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(thresholds=0),
            tf.keras.metrics.Recall(thresholds=0),
            tf.keras.metrics.BinaryIoU(target_class_ids=[0, 1], threshold=0.0),
            tf.keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.0)
        ]
    )

    # Evaluate model on test set
    results = model.evaluate(X_test, y_test, verbose=0)
    accuracy, precision, recall, iou_all, iou_class1 = results[1:]

    # Compute additional metrics (F1-score & AUC)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_binary = (y_pred > 0.5).astype(np.uint8)  # Convert to binary

    f1 = f1_score(y_test.flatten(), y_pred_binary.flatten())
    auc = roc_auc_score(y_test.flatten(), y_pred.flatten())

    # Store results
    experiment_results.append({
        "Experiment": experiment,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "IoU (All Classes)": iou_all,
        "IoU (Class 1)": iou_class1,
        "F1-score": f1,
        "AUC": auc
    })

# Convert results to a DataFrame
df_results = pd.DataFrame(experiment_results)
df_results = df_results.sort_values(by="Accuracy", ascending=False)  # Sort by accuracy

# Display results
print(df_results)

# Plot results
df_results.set_index("Experiment")[["Accuracy", "F1-score", "AUC"]].plot(kind="bar", figsize=(12, 6))
plt.title("Comparison of Experiment Results")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.legend(loc="lower right")
plt.show()



## Visualize Test Sample Predictions

In [ ]:
import random
import matplotlib.pyplot as plt

# Select a few random test images to visualize
num_samples = 5
random_indices = random.sample(range(len(X_test)), num_samples)

fig, axes = plt.subplots(num_samples, 3, figsize=(10, num_samples * 3))

for i, idx in enumerate(random_indices):
    image = X_test[idx]
    true_mask = y_test[idx]

    # Get predictions from the best-performing model
    best_model_name = df_results.iloc[0]["Experiment"]
    best_model_path = os.path.join(base_dir, best_model_name, "unet.keras")
    best_model = load_model(best_model_path, compile=False)
    predicted_mask = best_model.predict(np.expand_dims(image, axis=0))[0]
    predicted_mask = (predicted_mask > 0.5).astype(np.uint8)  # Binarize

    # Plot images
    axes[i, 0].imshow(image)
    axes[i, 0].set_title("Input Image")

    axes[i, 1].imshow(true_mask, cmap="gray")
    axes[i, 1].set_title("True Mask")

    axes[i, 2].imshow(predicted_mask, cmap="gray")
    axes[i, 2].set_title("Predicted Mask")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()


## Commit notebook to dagshub

In [ ]:
commit_message = ""

In [ ]:
from dagshub.notebook import save_notebook

repo = "chengzwk/omdena-frankfurt-ugs-unet"
branch = "regularize-experiments"
notebook_path = "regularize_experiment_results_summary.ipynb"

save_notebook(
    repo=repo,
    branch=branch,
    path=notebook_path,
    commit_message=commit_message,
    versioning="git"
    )